# Каскад рекомендаций: кандидаты, признаки и CatBoost-ранжирование

ML Mentor · starter notebook · решения в репозитории отсутствуют.


## Что предстоит сделать

За 120 минут вы соберёте полный рекомендательный каскад: три источника кандидатов, point-in-time признаки, эвристическое объединение и `CatBoostRanker`. Вы отдельно измерите потери на этапе кандидатов и качество итогового порядка. Обязательный CPU-прогон рассчитан не более чем на 15 минут.

**Итог:** схема каскада, таблица качества каждого слоя, разбор потерянных целей и решение о размере пула. Непоказанный объект нельзя называть доказанным негативом: это лишь неизвестное взаимодействие в implicit feedback.

### На чём основана работа

Структура эксперимента опирается на практику [недели 2](https://github.com/yandexdataschool/recsys_course/blob/2026_spring/week02_candgen/practice.ipynb) и [недели 3](https://github.com/yandexdataschool/recsys_course/blob/2026_spring/week03_ranking/practice/practice.ipynb) курса ШАД RecSys 2026. Формулировки, код и проверки ML Mentor написаны заново. Используется тот же закреплённый Apache-2.0 fixture Yambda, что и в предыдущей работе.

In [ ]:
import hashlib
import subprocess
import sys
import time
import urllib.request
from importlib.metadata import PackageNotFoundError, version as installed_version
from pathlib import Path
from urllib.parse import urlparse


REQUIRED_PACKAGES = {
    "pyarrow": "22.0.0",
    "implicit": "0.7.3",
    "catboost": "1.2.10",
}


def install_missing_packages() -> None:
    missing = []
    for package, version in REQUIRED_PACKAGES.items():
        try:
            actual = installed_version(package)
        except PackageNotFoundError:
            missing.append(f"{package}=={version}")
            continue
        if actual != version:
            missing.append(f"{package}=={version}")
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)


install_missing_packages()

import numpy as np
import pandas as pd
import pyarrow
import implicit
import catboost
from scipy import sparse


SEED = 42
FIXTURE_URL = "https://raw.githubusercontent.com/Vosatorp/ml-mentor-labs/a8542e1f42cecc21a1a7278955b6fd323ced8c9b/data/yambda/yambda-50m-likes-compact.parquet"
FIXTURE_SHA256 = "ca4dbdffea7114131ad9da411419838ed9c0a830613de935deaeecaad14d4805"
FIXTURE_BYTES = 2031420
EXPECTED_COLUMNS = ["uid", "item_id", "timestamp", "is_organic"]


def sha256(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def require_sha256(payload: bytes, expected: str, label: str) -> None:
    actual = sha256(payload)
    if actual != expected:
        raise RuntimeError(f"{label}: checksum changed: {actual}")


def require_final_hostname(response, expected: str) -> None:
    actual = urlparse(response.geturl()).hostname
    if actual != expected:
        raise RuntimeError(f"unexpected redirect host: {actual}")


def resolve_fixture_path() -> Path:
    for root in [Path.cwd(), *Path.cwd().parents]:
        candidate = root / "data" / "yambda" / "yambda-50m-likes-compact.parquet"
        if candidate.is_file():
            payload = candidate.read_bytes()
            if len(payload) != FIXTURE_BYTES:
                raise RuntimeError(f"fixture size changed: {len(payload)}")
            require_sha256(payload, FIXTURE_SHA256, "local Yambda fixture")
            return candidate

    cache_path = Path("/tmp/ml-mentor-labs-datasets-v1/yambda/yambda-50m-likes-compact.parquet")
    if cache_path.is_file():
        payload = cache_path.read_bytes()
        if len(payload) != FIXTURE_BYTES:
            raise RuntimeError(f"cached fixture size changed: {len(payload)}")
        require_sha256(payload, FIXTURE_SHA256, "cached Yambda fixture")
        return cache_path

    request = urllib.request.Request(FIXTURE_URL, headers={"User-Agent": "ML-Mentor-Labs/1.0"})
    with urllib.request.urlopen(request, timeout=120) as response:
        require_final_hostname(response, "raw.githubusercontent.com")
        payload = response.read()
    if len(payload) != FIXTURE_BYTES:
        raise RuntimeError(f"downloaded fixture size changed: {len(payload)}")
    require_sha256(payload, FIXTURE_SHA256, "downloaded Yambda fixture")
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache_path.write_bytes(payload)
    return cache_path


FIXTURE_PATH = resolve_fixture_path()
events = pd.read_parquet(FIXTURE_PATH, columns=EXPECTED_COLUMNS)
events = events.sort_values(["timestamp", "uid", "item_id"], kind="mergesort").reset_index(drop=True)
if list(events.columns) != EXPECTED_COLUMNS:
    raise RuntimeError(f"unexpected columns: {list(events.columns)}")
if len(events) != 268_631 or events.uid.nunique() != 1_999 or events.item_id.nunique() != 42_965:
    raise RuntimeError("unexpected fixture statistics")
if not events.timestamp.is_monotonic_increasing:
    raise RuntimeError("events must be sorted by time")

TRAIN_BOUNDARY = int(events.timestamp.quantile(0.70, interpolation="lower"))
VALIDATION_BOUNDARY = int(events.timestamp.quantile(0.85, interpolation="lower"))
train_events = events[events.timestamp <= TRAIN_BOUNDARY].copy()
validation_events = events[(events.timestamp > TRAIN_BOUNDARY) & (events.timestamp <= VALIDATION_BOUNDARY)].copy()
test_events = events[events.timestamp > VALIDATION_BOUNDARY].copy()

USER_IDS = np.sort(train_events.uid.unique())
ITEM_IDS = np.sort(train_events.item_id.unique())
USER_TO_INDEX = {int(value): index for index, value in enumerate(USER_IDS)}
ITEM_TO_INDEX = {int(value): index for index, value in enumerate(ITEM_IDS)}
INDEX_TO_ITEM = np.asarray(ITEM_IDS)


def to_user_item_matrix(frame: pd.DataFrame) -> sparse.csr_matrix:
    known = frame[frame.uid.isin(USER_TO_INDEX) & frame.item_id.isin(ITEM_TO_INDEX)]
    rows = known.uid.map(USER_TO_INDEX).to_numpy()
    columns = known.item_id.map(ITEM_TO_INDEX).to_numpy()
    values = np.ones(len(known), dtype=np.float32)
    matrix = sparse.csr_matrix((values, (rows, columns)), shape=(len(USER_IDS), len(ITEM_IDS)))
    matrix.data[:] = 1.0
    matrix.eliminate_zeros()
    return matrix


train_matrix = to_user_item_matrix(train_events)
print({
    "versions": {"pyarrow": pyarrow.__version__, "implicit": implicit.__version__, "catboost": catboost.__version__},
    "events": len(events),
    "train": len(train_events),
    "validation": len(validation_events),
    "test": len(test_events),
    "users_in_train": train_matrix.shape[0],
    "items_in_train": train_matrix.shape[1],
})


## 1. Два временных снимка

Для обучения ранжировщика история заканчивается на первой границе, а целевые лайки берутся из validation. Для финальной проверки кандидаты и признаки строятся по истории `train + validation`, цели — из test. Нельзя один раз посчитать признаки по полному логу и потом разрезать строки: так в train попадёт будущее.

Сформируйте таблицу запросов `uid, query_timestamp` для обоих снимков. Ограничьте обязательный прогон 1 000 подходящих пользователей, выбранных детерминированно по SHA-256 от `uid`, чтобы время CPU оставалось предсказуемым.

In [ ]:
def deterministic_users(user_ids, limit=1_000):
    # TODO: отсортируйте uid по SHA-256 строки recsys-cascade-v1:<uid> и возьмите limit.
    raise NotImplementedError


def build_snapshot(history: pd.DataFrame, targets: pd.DataFrame, user_ids):
    # TODO: верните history, targets и queries. query_timestamp строго больше max(history.timestamp)
    # и не больше min(targets.timestamp) для каждого пользователя.
    raise NotImplementedError


ranker_users = deterministic_users(
    sorted(set(train_events.uid) & set(validation_events.uid) & set(test_events.uid)),
    limit=1_000,
)
training_snapshot = build_snapshot(train_events, validation_events, ranker_users)
test_history = pd.concat([train_events, validation_events], ignore_index=True)
evaluation_snapshot = build_snapshot(test_history, test_events, ranker_users)


## 2. Три источника кандидатов

Для каждого снимка обучите популярность, item-to-item и ALS только на его истории. Каждый источник возвращает top-100 после удаления просмотренных объектов. Сохраните исходный `score` и `rank`: после объединения они станут признаками и позволят понять, какой генератор нашёл цель.

Здесь можно переиспользовать функции из предыдущей лабораторной, но notebook должен оставаться исполняемым сам по себе.

In [ ]:
from implicit.als import AlternatingLeastSquares
from implicit.nearest_neighbours import CosineRecommender


SOURCES = ("popularity", "item_to_item", "als")
SOURCE_K = 100


def fit_candidate_models(history: pd.DataFrame):
    # TODO: обучите три источника на history и верните модели, матрицу и отображения ID.
    raise NotImplementedError


def generate_source_candidates(bundle, queries: pd.DataFrame, k=SOURCE_K):
    # TODO: верните long DataFrame columns=[uid, item_id, source, source_score, source_rank].
    # Для каждого uid и каждого источника не более k строк; просмотренные объекты исключены.
    raise NotImplementedError


training_bundle = fit_candidate_models(training_snapshot["history"])
training_source_candidates = generate_source_candidates(training_bundle, training_snapshot["queries"])
evaluation_bundle = fit_candidate_models(evaluation_snapshot["history"])
evaluation_source_candidates = generate_source_candidates(evaluation_bundle, evaluation_snapshot["queries"])


## 3. Объединение и потери кандидатов

Дедуплицируйте пары `(uid, item_id)`, но не теряйте происхождение: для каждого источника нужны отдельные score, rank и бинарный признак присутствия. Рассчитайте candidate Recall@100 для каждого источника и их объединения. Это верхняя граница для ранжировщика: он не вернёт цель, которой нет в пуле.

In [ ]:
def merge_candidates(source_candidates: pd.DataFrame) -> pd.DataFrame:
    # TODO: одна строка на uid,item_id; columns для каждого источника:
    # <source>_score, <source>_rank, from_<source>. Отсутствующий rank заполните SOURCE_K + 1.
    raise NotImplementedError


def candidate_recall(source_candidates, targets, k=100):
    # TODO: посчитайте macro Recall@k по пользователям отдельно для источников и union.
    raise NotImplementedError


training_pool = merge_candidates(training_source_candidates)
evaluation_pool = merge_candidates(evaluation_source_candidates)
training_candidate_report = candidate_recall(training_source_candidates, training_snapshot["targets"], k=100)
evaluation_candidate_report = candidate_recall(evaluation_source_candidates, evaluation_snapshot["targets"], k=100)


## 4. Point-in-time признаки

Для каждой пары пользователь–кандидат посчитайте признаки только по истории снимка: длину истории пользователя, популярность объекта, давность последнего лайка пользователя, встречался ли объект среди последних 10 лайков, score/rank источников и число источников. `is_organic` можно агрегировать только по прошлым событиям.

Метка равна 1, если объект встречается среди целей этого запроса, иначе 0. Ноль здесь — не доказанный дизлайк, а рабочее приближение для обучения порядка внутри уже сформированного пула.

In [ ]:
FEATURE_COLUMNS = [
    "user_history_size", "item_popularity", "user_recency_seconds",
    "in_last_10", "organic_share", "source_count",
    "popularity_score", "popularity_rank",
    "item_to_item_score", "item_to_item_rank",
    "als_score", "als_rank",
]


def point_in_time_features(pool, history, targets, queries):
    # TODO: верните pool + FEATURE_COLUMNS + label + group_id.
    # Для каждой строки докажите history.timestamp < query_timestamp.
    raise NotImplementedError


training_rows = point_in_time_features(
    training_pool, training_snapshot["history"], training_snapshot["targets"], training_snapshot["queries"]
)
evaluation_rows = point_in_time_features(
    evaluation_pool, evaluation_snapshot["history"], evaluation_snapshot["targets"], evaluation_snapshot["queries"]
)


## 5. Эвристика и CatBoostRanker

Сначала постройте прозрачную эвристику reciprocal rank fusion: сумма `1 / (60 + rank)` по источникам. Затем обучите `CatBoostRanker` с `loss_function="YetiRankPairwise"`, `iterations=150`, `depth=6`, `learning_rate=0.08`, `random_seed=42`, `verbose=False`, `thread_count=-1`. Перед fit отсортируйте строки по `group_id`, чтобы строки одного пользовательского запроса шли подряд.

Не подбирайте параметры по test. Если в обучающем запросе нет ни одного положительного кандидата, исключите всю группу и посчитайте долю таких потерь отдельно.

In [ ]:
from catboost import CatBoostRanker


def reciprocal_rank_fusion(frame: pd.DataFrame) -> np.ndarray:
    # TODO: верните score по трём source rank; отсутствующий источник не даёт вклад.
    raise NotImplementedError


def fit_ranker(training_frame: pd.DataFrame):
    # TODO: удалите группы без positive, отсортируйте по group_id и обучите закреплённый ranker.
    # Верните model, fit_seconds и долю исключённых запросов.
    raise NotImplementedError


training_rows["heuristic_score"] = reciprocal_rank_fusion(training_rows)
evaluation_rows["heuristic_score"] = reciprocal_rank_fusion(evaluation_rows)
ranker, ranker_fit_seconds, dropped_training_group_share = fit_ranker(training_rows)
evaluation_rows["ranker_score"] = ranker.predict(evaluation_rows[FEATURE_COLUMNS])


## 6. Качество слоёв, задержка и размер пула

Сравните эвристику и ranker по Recall@10, NDCG@10, покрытию каталога и времени сортировки. Рядом покажите candidate Recall@100: если он низкий, перестановка кандидатов проблему не исправит. Повторите оценку union для размеров источника 20, 50 и 100 и выберите минимальный пул, который укладывается в допустимую потерю качества и задержку.

В письменном выводе разделите: потери генерации, потери ранжирования, ограничение implicit feedback и production-стоимость. Укажите fallback при недоступности одного источника или ranker.

In [ ]:
def ranking_report(frame, targets, score_column, catalog_size, k=10):
    # TODO: macro Recall@k, NDCG@k, catalog coverage и mean sorting milliseconds per user.
    raise NotImplementedError


heuristic_report = ranking_report(
    evaluation_rows, evaluation_snapshot["targets"], "heuristic_score", evaluation_bundle["matrix"].shape[1], k=10
)
ranker_report = ranking_report(
    evaluation_rows, evaluation_snapshot["targets"], "ranker_score", evaluation_bundle["matrix"].shape[1], k=10
)
layer_report = pd.DataFrame([
    *evaluation_candidate_report.to_dict("records"),
    {"layer": "heuristic_top10", **heuristic_report},
    {"layer": "catboost_ranker_top10", **ranker_report},
])


def pool_size_experiment(source_candidates, targets, sizes=(20, 50, 100)):
    # TODO: для каждого размера источника верните union candidate recall, mean pool size и build milliseconds/user.
    raise NotImplementedError


pool_size_report = pool_size_experiment(evaluation_source_candidates, evaluation_snapshot["targets"])
CASCADE_DECISION = {
    "source_pool_size": None,  # TODO: 20 | 50 | 100
    "candidate_loss": None,
    "ranking_loss": None,
    "implicit_feedback_limit": None,
    "latency_tradeoff": None,
    "fallback": None,
}


In [ ]:
# Self-check: проверяет границы эксперимента, но не подставляет вывод.
assert len(ranker_users) == 1_000 and len(set(ranker_users)) == 1_000
assert training_snapshot["history"].timestamp.max() < training_snapshot["targets"].timestamp.min()
assert evaluation_snapshot["history"].timestamp.max() < evaluation_snapshot["targets"].timestamp.min()
required_source_columns = {"uid", "item_id", "source", "source_score", "source_rank"}
assert required_source_columns <= set(training_source_candidates)
assert set(training_source_candidates.source) == set(SOURCES)
assert not training_pool.duplicated(["uid", "item_id"]).any()
assert not evaluation_pool.duplicated(["uid", "item_id"]).any()
assert set(FEATURE_COLUMNS) <= set(training_rows) and set(FEATURE_COLUMNS) <= set(evaluation_rows)
assert np.isfinite(training_rows[FEATURE_COLUMNS].to_numpy()).all()
assert training_rows.groupby("group_id").label.max().isin([0, 1]).all()
assert 0 <= dropped_training_group_share < 1
assert {"popularity", "item_to_item", "als", "union"} <= set(evaluation_candidate_report.layer)
assert {20, 50, 100} == set(pool_size_report.source_k)
assert all(CASCADE_DECISION[key] is not None for key in CASCADE_DECISION)
assert "неизвест" in CASCADE_DECISION["implicit_feedback_limit"].lower()
display(layer_report)
display(pool_size_report)


## Что сдать

1. Заполненный notebook и схему `история → три источника → union → признаки → ranker → top-10`.
2. Candidate Recall@100 по каждому источнику и union.
3. Recall@10, NDCG@10, покрытие и задержку эвристики и ranker.
4. Эксперимент с пулами 20/50/100 и решение о размере пула.
5. Объяснение, почему непоказанные объекты — неизвестные взаимодействия, а не доказанные негативы.

Не загружайте решения в публичный репозиторий ML Mentor. Личную выполненную копию храните в Google Drive или локально.